In [1]:
# ============================================================
# 07_model_comparison.ipynb
# Benchmark: XGBoost vs LightGBM vs Random Forest
#
# Same pipeline as 02_ml_underwriting:
#   Optuna HPO + 5-fold inner CV + stratified 80/20 split
# Total subgroup only (male/female by disease = 9 models each
# would be excessive; total is sufficient for benchmarking)
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Imports
# ─────────────────────────────────────────────
import os
import json
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (
    StratifiedKFold, train_test_split
)
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE_OUT = os.path.join('..', 'outputs')
BASE_MOD = os.path.join('..', 'outputs', 'models')
os.makedirs(BASE_MOD, exist_ok=True)

RANDOM_SEED = 42
N_TRIALS    = 50
N_CV        = 5

print("Libraries loaded.")
print(f"  XGBoost   : {xgb.__version__}")
print(f"  LightGBM  : {lgb.__version__}")


# ─────────────────────────────────────────────
# Cell 2 | Registry
# ─────────────────────────────────────────────
DISEASE_REGISTRY = {
    'htn': {
        'target'      : 'HE_HP',
        'label'       : 'Hypertension',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_NA', 'N_K'],
    },
    'dm': {
        'target'      : 'HE_DM_HbA1c',
        'label'       : 'Diabetes',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_SUGAR', 'N_CHO', 'N_EN'],
    },
    'dys': {
        'target'      : 'HE_HCHOL',
        'label'       : 'Dyslipidemia',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_FAT', 'N_CHOL'],
    },
}

NON_ACTIONABLE = [
    'ID', 'year', 'age_group', 'age', 'sex',
    'edu', 'incm', 'ho_incm',
    'BE3_71', 'BE3_81', 'BP1', 'mh_stress',
    'HE_obe', 'BO1_1', 'BO1_2', 'BO1_3',
]

def load_Xy(d_key, d_info):
    df = pd.read_csv(
        os.path.join(BASE_OUT, f"{d_key}_total.csv")
    )
    target = d_info['target']
    feat   = d_info['feature_cols']
    drop   = NON_ACTIONABLE + [target]
    fcols  = [c for c in df.columns
               if c not in drop and c in feat]
    X = df[fcols].astype(float)
    y = df[target].astype(int)
    return X, y, fcols


# ─────────────────────────────────────────────
# Cell 3 | Evaluation Helper
# ─────────────────────────────────────────────
def evaluate(model, X_test, y_test):
    prob = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    auroc = roc_auc_score(y_test, prob)
    auprc = average_precision_score(y_test, prob)
    brier = brier_score_loss(y_test, prob)
    cm    = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    sens  = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec  = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        'AUROC': round(auroc, 4),
        'AUPRC': round(auprc, 4),
        'Brier': round(brier, 4),
        'Sensitivity': round(sens, 4),
        'Specificity': round(spec, 4),
    }


# ─────────────────────────────────────────────
# Cell 4 | Optuna Objectives
# ─────────────────────────────────────────────
def objective_xgb(trial, X_tr, y_tr, spw):
    params = {
        'verbosity'        : 0,
        'objective'        : 'binary:logistic',
        'eval_metric'      : 'auc',
        'tree_method'      : 'hist',
        'random_state'     : RANDOM_SEED,
        'scale_pos_weight' : spw,
        'n_estimators'     : trial.suggest_int(
                                 'n_estimators',
                                 200, 1000, step=100),
        'max_depth'        : trial.suggest_int(
                                 'max_depth', 3, 8),
        'learning_rate'    : trial.suggest_float(
                                 'learning_rate',
                                 0.01, 0.2, log=True),
        'subsample'        : trial.suggest_float(
                                 'subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float(
                                 'colsample_bytree',
                                 0.6, 1.0),
        'min_child_weight' : trial.suggest_int(
                                 'min_child_weight',
                                 1, 10),
        'reg_alpha'        : trial.suggest_float(
                                 'reg_alpha',
                                 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float(
                                 'reg_lambda',
                                 1e-4, 10.0, log=True),
        'gamma'            : trial.suggest_float(
                                 'gamma', 0.0, 5.0),
    }
    skf  = StratifiedKFold(n_splits=N_CV, shuffle=True,
                           random_state=RANDOM_SEED)
    aucs = []
    for tr_i, va_i in skf.split(X_tr, y_tr):
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i],
              eval_set=[(X_tr.iloc[va_i],
                         y_tr.iloc[va_i])],
              verbose=False)
        p = m.predict_proba(X_tr.iloc[va_i])[:, 1]
        aucs.append(roc_auc_score(y_tr.iloc[va_i], p))
    return np.mean(aucs)


def objective_lgb(trial, X_tr, y_tr, spw):
    params = {
        'verbosity'       : -1,
        'objective'       : 'binary',
        'metric'          : 'auc',
        'random_state'    : RANDOM_SEED,
        'is_unbalance'    : False,
        'scale_pos_weight': spw,
        'n_estimators'    : trial.suggest_int(
                                'n_estimators',
                                200, 1000, step=100),
        'max_depth'       : trial.suggest_int(
                                'max_depth', 3, 8),
        'learning_rate'   : trial.suggest_float(
                                'learning_rate',
                                0.01, 0.2, log=True),
        'subsample'       : trial.suggest_float(
                                'subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float(
                                'colsample_bytree',
                                0.6, 1.0),
        'min_child_samples': trial.suggest_int(
                                'min_child_samples',
                                5, 50),
        'reg_alpha'       : trial.suggest_float(
                                'reg_alpha',
                                1e-4, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float(
                                'reg_lambda',
                                1e-4, 10.0, log=True),
        'num_leaves'      : trial.suggest_int(
                                'num_leaves', 15, 127),
    }
    skf  = StratifiedKFold(n_splits=N_CV, shuffle=True,
                           random_state=RANDOM_SEED)
    aucs = []
    for tr_i, va_i in skf.split(X_tr, y_tr):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i],
              eval_set=[(X_tr.iloc[va_i],
                         y_tr.iloc[va_i])],
              callbacks=[lgb.early_stopping(
                  50, verbose=False),
                  lgb.log_evaluation(-1)])
        p = m.predict_proba(X_tr.iloc[va_i])[:, 1]
        aucs.append(roc_auc_score(y_tr.iloc[va_i], p))
    return np.mean(aucs)


def objective_rf(trial, X_tr, y_tr, spw):
    params = {
        'random_state'    : RANDOM_SEED,
        'class_weight'    : {0: 1, 1: spw},
        'n_jobs'          : -1,
        'n_estimators'    : trial.suggest_int(
                                'n_estimators',
                                200, 800, step=100),
        'max_depth'       : trial.suggest_int(
                                'max_depth', 3, 15),
        'min_samples_leaf': trial.suggest_int(
                                'min_samples_leaf',
                                1, 20),
        'max_features'    : trial.suggest_float(
                                'max_features',
                                0.3, 1.0),
    }
    skf  = StratifiedKFold(n_splits=N_CV, shuffle=True,
                           random_state=RANDOM_SEED)
    aucs = []
    for tr_i, va_i in skf.split(X_tr, y_tr):
        m = RandomForestClassifier(**params)
        m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i])
        p = m.predict_proba(X_tr.iloc[va_i])[:, 1]
        aucs.append(roc_auc_score(y_tr.iloc[va_i], p))
    return np.mean(aucs)


# ─────────────────────────────────────────────
# Cell 5 | Main Comparison Loop
# ─────────────────────────────────────────────
MODEL_CONFIGS = {
    'XGBoost'     : objective_xgb,
    'LightGBM'    : objective_lgb,
    'RandomForest': objective_rf,
}

MODEL_CLASSES = {
    'XGBoost'     : xgb.XGBClassifier,
    'LightGBM'    : lgb.LGBMClassifier,
    'RandomForest': RandomForestClassifier,
}

# Fixed kwargs added after Optuna (not in search space)
MODEL_FIXED = {
    'XGBoost': {
        'verbosity': 0,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'random_state': RANDOM_SEED,
    },
    'LightGBM': {
        'verbosity': -1,
        'objective': 'binary',
        'random_state': RANDOM_SEED,
    },
    'RandomForest': {
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
    },
}

all_cmp = []

for d_key, d_info in DISEASE_REGISTRY.items():
    label = d_info['label']
    print(f"\n{'='*58}")
    print(f"  {label}")
    print(f"{'='*58}")

    X, y, fcols = load_Xy(d_key, d_info)
    X_train, X_test, y_train, y_test = \
        train_test_split(
            X, y, test_size=0.2,
            stratify=y, random_state=RANDOM_SEED
        )

    n0  = (y_train == 0).sum()
    n1  = (y_train == 1).sum()
    spw = round(n0 / n1, 4)
    print(f"  Train={len(X_train):,}  "
          f"Test={len(X_test):,}  "
          f"spw={spw}")

    for model_name, obj_fn in MODEL_CONFIGS.items():
        print(f"\n  [{model_name}] "
              f"Optuna {N_TRIALS} trials ...")

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(
                seed=RANDOM_SEED)
        )
        study.optimize(
            lambda trial: obj_fn(
                trial, X_train, y_train, spw),
            n_trials=N_TRIALS,
            show_progress_bar=False
        )

        best_p = study.best_params.copy()
        best_p.update(MODEL_FIXED[model_name])

        # Add spw for XGB/LGB
        if model_name == 'XGBoost':
            best_p['scale_pos_weight'] = spw
        elif model_name == 'LightGBM':
            best_p['scale_pos_weight'] = spw
        elif model_name == 'RandomForest':
            best_p['class_weight'] = {0: 1, 1: spw}

        final = MODEL_CLASSES[model_name](**best_p)
        final.fit(X_train, y_train)

        metrics = evaluate(final, X_test, y_test)
        metrics.update({
            'Disease'   : label,
            'Model'     : model_name,
            'CV_AUROC'  : round(study.best_value, 4),
            'N_train'   : len(X_train),
            'N_test'    : len(X_test),
        })
        all_cmp.append(metrics)

        print(f"    CV AUROC={study.best_value:.4f}  "
              f"Test AUROC={metrics['AUROC']:.4f}  "
              f"AUPRC={metrics['AUPRC']:.4f}  "
              f"Brier={metrics['Brier']:.4f}")

        # Save model
        joblib.dump(final, os.path.join(
            BASE_MOD,
            f"{model_name.lower()}_{d_key}_total.joblib"
        ))


# ─────────────────────────────────────────────
# Cell 6 | Comparison Table
# ─────────────────────────────────────────────
df_cmp = pd.DataFrame(all_cmp)

print("\n" + "="*80)
print("  MODEL COMPARISON SUMMARY")
print("="*80)

# Pivot: Disease × Model → metrics
for disease in [d['label']
                for d in DISEASE_REGISTRY.values()]:
    sub = df_cmp[df_cmp['Disease'] == disease]\
                [['Model', 'CV_AUROC', 'AUROC',
                  'AUPRC', 'Brier',
                  'Sensitivity', 'Specificity']]
    print(f"\n  {disease}:")
    print(sub.to_string(index=False))

cmp_path = os.path.join(BASE_OUT,
                         'model_comparison.csv')
df_cmp.to_csv(cmp_path, index=False)
print(f"\nComparison table saved : {cmp_path}")
print("\n=== 07_model_comparison.ipynb complete ===")

C:\Users\miy\miniconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.
  XGBoost   : 1.7.5
  LightGBM  : 4.6.0

  Hypertension
  Train=11,488  Test=2,873  spw=2.104

  [XGBoost] Optuna 50 trials ...
    CV AUROC=0.7826  Test AUROC=0.7779  AUPRC=0.6187  Brier=0.1925

  [LightGBM] Optuna 50 trials ...
    CV AUROC=0.7823  Test AUROC=0.7759  AUPRC=0.6088  Brier=0.1923

  [RandomForest] Optuna 50 trials ...
    CV AUROC=0.7769  Test AUROC=0.7751  AUPRC=0.6005  Brier=0.1894

  Diabetes
  Train=10,815  Test=2,704  spw=3.1532

  [XGBoost] Optuna 50 trials ...
    CV AUROC=0.8474  Test AUROC=0.8511  AUPRC=0.5885  Brier=0.1616

  [LightGBM] Optuna 50 trials ...
    CV AUROC=0.8463  Test AUROC=0.8485  AUPRC=0.5863  Brier=0.1608

  [RandomForest] Optuna 50 trials ...
    CV AUROC=0.8390  Test AUROC=0.8496  AUPRC=0.5874  Brier=0.1536

  Dyslipidemia
  Train=17,025  Test=4,257  spw=2.2988

  [XGBoost] Optuna 50 trials ...
    CV AUROC=0.7047  Test AUROC=0.7119  AUPRC=0.4828  Brier=0.2163

  [LightGBM] Optuna 50 trials ...
    CV AUROC=0.7038  Test AU